# Castalia Scholar — The Report Generator (Writing Agent)

## What This Notebook Teaches You

This is **Part 2 of the Castalia Scholar capstone project**. In Notebook 34, we built the Research Agent that finds, retrieves, and extracts information. Now we build the **Writing Agent** — the component that takes analyzed research results and produces a well-structured, citation-backed research report.

Writing is more than stringing sentences together. A good research report requires:
- **Structure** — logical section ordering with clear purpose for each part
- **Evidence** — every claim supported by citations from analyzed sources
- **Coherence** — sections flow naturally, terms are consistent, no redundancy
- **Quality** — clear prose, appropriate depth, proper academic tone

**Techniques used from previous notebooks**:
- Plan-and-Execute (Notebook 06) — for section planning
- Reflection & Self-Critique (Notebook 07) — for iterative quality improvement
- Tool Use (Notebook 02) — citation manager as a tool
- Structured Output (Notebook 03) — for report templates

By the end of this notebook, you will:

1. **Build report data structures** — templates, sections, citations as dataclasses
2. **Build a SectionPlanner** — map analysis findings to a report outline
3. **Build a SectionWriter** — write individual sections with evidence and transitions
4. **Build a CitationManager** — track and format academic citations
5. **Build a CoherenceChecker** — detect redundancy, logical gaps, and inconsistency
6. **Build the full WritingAgent** — orchestrate planning, writing, citing, and refining
7. **Generate a complete report** from mock analysis data
8. **Measure quality** — completeness, citation coverage, coherence, readability

---

> **Prerequisites**: Notebooks 01–09, 34 (Research Agent).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~60–75 minutes.

## Part 0: Environment Setup

Same Qwen3-8B setup. The model will act as a structured writer — generating section content, formatting citations, and checking coherence.

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. The Writing Agent's Role in Castalia Scholar

### 1.1 — System Architecture Recap

```
  User Query
      │
      ▼
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│   Research    │────►│   Analysis   │────►│   Writing    │
│    Agent      │     │    Agent     │     │    Agent     │  ← THIS notebook
│ (Notebook 34) │     │  (built-in)  │     │ (Notebook 35) │
└──────────────┘     └──────────────┘     └──────────────┘
      │                    │                      │
   Sources            Findings              Final Report
   extracted          synthesized           with citations
```

### 1.2 — What the Writing Agent Receives

From the Analysis Agent, the Writing Agent receives:
- **Findings**: key claims and insights, each tied to source(s)
- **Sources**: metadata about each source (author, title, year, URL)
- **Contradictions**: where sources disagree
- **Confidence scores**: how well-supported each finding is

### 1.3 — What the Writing Agent Produces

A structured research report with:
- Title and abstract
- Introduction with context
- Background and related work
- Findings organized by theme
- Analysis with comparison tables
- Conclusion with implications
- References section with formatted citations

## 2. Report Structure — Data Models

We define the report structure as dataclasses. This gives us type safety, easy serialization, and clear contracts between components.

In [ ]:
import textwrap

@dataclass
class Source:
    """A research source with citation metadata."""
    source_id: str
    title: str
    authors: List[str]
    year: int
    url: str = ""
    source_type: str = "article"  # article, book, web, paper

    def short_cite(self) -> str:
        """Short in-text citation: [Author, Year]"""
        first_author = self.authors[0].split()[-1] if self.authors else "Unknown"
        et_al = " et al." if len(self.authors) > 2 else (
            f" & {self.authors[1].split()[-1]}" if len(self.authors) == 2 else "")
        return f"[{first_author}{et_al}, {self.year}]"

    def full_reference(self) -> str:
        """Full reference entry for bibliography."""
        author_str = ", ".join(self.authors)
        ref = f"{author_str} ({self.year}). {self.title}."
        if self.url:
            ref += f" Retrieved from {self.url}"
        return ref

@dataclass
class Finding:
    """A research finding tied to source(s)."""
    finding_id: str
    claim: str
    evidence: str
    source_ids: List[str]
    confidence: float = 0.8
    theme: str = "general"

@dataclass
class AnalysisOutput:
    """The output from the Analysis Agent — input to the Writing Agent."""
    topic: str
    findings: List[Finding]
    sources: List[Source]
    contradictions: List[Dict[str, Any]] = field(default_factory=list)

    def get_source(self, source_id: str) -> Optional[Source]:
        for s in self.sources:
            if s.source_id == source_id:
                return s
        return None

@dataclass
class ReportSection:
    """A single section of the research report."""
    heading: str
    content: str = ""
    subsections: List['ReportSection'] = field(default_factory=list)
    citations_used: List[str] = field(default_factory=list)  # source_ids

@dataclass
class ReportTemplate:
    """A complete research report."""
    title: str
    abstract: str = ""
    sections: List[ReportSection] = field(default_factory=list)
    citations: List[str] = field(default_factory=list)  # all source_ids used
    metadata: Dict[str, Any] = field(default_factory=dict)

    def word_count(self) -> int:
        total = len(self.abstract.split())
        for s in self.sections:
            total += len(s.content.split())
            for sub in s.subsections:
                total += len(sub.content.split())
        return total

    def render(self) -> str:
        """Render the report as plain text."""
        lines = [f"# {self.title}", "", self.abstract, ""]
        for section in self.sections:
            lines.append(f"## {section.heading}")
            lines.append(section.content)
            lines.append("")
            for sub in section.subsections:
                lines.append(f"### {sub.heading}")
                lines.append(sub.content)
                lines.append("")
        return "\n".join(lines)

# Quick test
s1 = Source("s1", "Chain-of-Thought Prompting Elicits Reasoning", ["Jason Wei", "Xuezhi Wang"], 2022)
s2 = Source("s2", "Self-Consistency Improves CoT Reasoning", ["Xuezhi Wang", "Jason Wei", "Dale Schuurmans"], 2023)
print(f"Short cite: {s1.short_cite()}")
print(f"Short cite: {s2.short_cite()}")
print(f"Full ref:   {s1.full_reference()}")
print(f"\n✓ Report data structures defined")

### 2.1 — Mock Analysis Data

For testing, we create realistic mock analysis output that simulates what the Analysis Agent (Notebook 34) would produce.

In [ ]:
def create_mock_analysis(topic: str) -> AnalysisOutput:
    """Create mock analysis data for testing the Writing Agent."""

    sources = [
        Source("s1", "Chain-of-Thought Prompting Elicits Reasoning in Large Language Models",
               ["Jason Wei", "Xuezhi Wang", "Dale Schuurmans"], 2022,
               "https://arxiv.org/abs/2201.11903", "paper"),
        Source("s2", "Self-Consistency Improves Chain of Thought Reasoning in Language Models",
               ["Xuezhi Wang", "Jason Wei", "Dale Schuurmans"], 2023,
               "https://arxiv.org/abs/2203.11171", "paper"),
        Source("s3", "Tree of Thoughts: Deliberate Problem Solving with LLMs",
               ["Shunyu Yao", "Dian Yu", "Jeffrey Zhao"], 2023,
               "https://arxiv.org/abs/2305.10601", "paper"),
        Source("s4", "Large Language Models are Zero-Shot Reasoners",
               ["Takeshi Kojima", "Shixiang Shane Gu"], 2022,
               "https://arxiv.org/abs/2205.11916", "paper"),
        Source("s5", "Measuring and Narrowing the Compositionality Gap in Language Models",
               ["Ofir Press", "Muru Zhang", "Sewon Min"], 2023,
               "https://arxiv.org/abs/2210.03350", "paper"),
    ]

    findings = [
        Finding("f1",
                "Chain-of-thought prompting significantly improves multi-step reasoning in LLMs",
                "Adding 'Let's think step by step' or providing few-shot CoT examples improved accuracy on GSM8K from 17.9% to 58.1% for PaLM 540B.",
                ["s1", "s4"], confidence=0.95, theme="effectiveness"),
        Finding("f2",
                "CoT benefits scale with model size — small models show limited improvement",
                "Models under 10B parameters showed minimal benefit from CoT prompting. The technique primarily helps models above 100B parameters.",
                ["s1"], confidence=0.85, theme="scaling"),
        Finding("f3",
                "Self-consistency with majority voting further improves CoT reasoning accuracy",
                "Sampling multiple reasoning paths and taking the majority answer improved GSM8K accuracy from 58.1% to 74.4%.",
                ["s2"], confidence=0.90, theme="effectiveness"),
        Finding("f4",
                "Tree-of-thought extends CoT by exploring branching reasoning paths",
                "ToT allows the model to consider multiple reasoning branches, backtrack, and select the best path. It solved 74% of Game of 24 tasks vs 4% for standard CoT.",
                ["s3"], confidence=0.88, theme="extensions"),
        Finding("f5",
                "Zero-shot CoT ('Let's think step by step') works surprisingly well without examples",
                "Simply adding the phrase achieved competitive performance with few-shot methods on many benchmarks.",
                ["s4"], confidence=0.82, theme="accessibility"),
        Finding("f6",
                "Compositional reasoning remains a challenge even with CoT",
                "On multi-hop questions requiring composing multiple facts, CoT helps but doesn't fully close the gap with human performance.",
                ["s5"], confidence=0.78, theme="limitations"),
    ]

    contradictions = [
        {
            "finding_ids": ["f1", "f2"],
            "description": "CoT is highly effective overall, but this benefit is concentrated in large models — it may not be universally applicable."
        },
        {
            "finding_ids": ["f5", "f6"],
            "description": "Zero-shot CoT works well on standard tasks, but compositional reasoning still challenges even prompted models."
        }
    ]

    return AnalysisOutput(
        topic=topic,
        findings=findings,
        sources=sources,
        contradictions=contradictions
    )

mock_data = create_mock_analysis("Chain-of-thought prompting and LLM reasoning")
print(f"Topic: {mock_data.topic}")
print(f"Sources: {len(mock_data.sources)}")
print(f"Findings: {len(mock_data.findings)}")
print(f"Contradictions: {len(mock_data.contradictions)}")
print(f"\nFindings by theme:")
themes = defaultdict(list)
for f in mock_data.findings:
    themes[f.theme].append(f.finding_id)
for theme, ids in themes.items():
    print(f"  {theme}: {ids}")

## 3. Section Planner — Mapping Findings to Report Structure

The SectionPlanner decides **what to write and where**. Given analysis findings, it creates an outline that maps each finding to the appropriate report section.

This uses the **Plan-and-Execute** pattern from Notebook 06 — plan first, then execute section by section.

In [ ]:
class SectionPlanner:
    """Plans the report structure based on analysis findings."""

    # Standard academic report sections
    STANDARD_SECTIONS = [
        "Introduction",
        "Background",
        "Key Findings",
        "Analysis and Discussion",
        "Limitations and Open Questions",
        "Conclusion",
    ]

    def plan(self, analysis: AnalysisOutput) -> ReportTemplate:
        """Create a report outline by mapping findings to sections."""
        # Use LLM to create a topic-aware outline
        finding_summary = "\n".join(
            f"- [{f.finding_id}] {f.claim} (theme: {f.theme}, confidence: {f.confidence})"
            for f in analysis.findings
        )
        contradiction_summary = "\n".join(
            f"- {c['description']}" for c in analysis.contradictions
        ) or "None identified."

        messages = [
            {"role": "system", "content": "You are a report planning assistant. Given research findings, create a brief outline that maps each finding to the most appropriate report section. Respond in this exact JSON format: {\"section_assignments\": {\"finding_id\": \"section_name\", ...}, \"suggested_title\": \"...\"}"},
            {"role": "user", "content": f"Topic: {analysis.topic}\n\nFindings:\n{finding_summary}\n\nContradictions:\n{contradiction_summary}\n\nStandard sections: {self.STANDARD_SECTIONS}\n\nAssign each finding to the best section and suggest a report title."}
        ]

        raw = generate(messages, max_new_tokens=300, temperature=0.3, do_sample=True)

        # Parse LLM response
        assignments = {}
        title = f"A Survey of {analysis.topic}"
        json_match = re.search(r'\{[\s\S]*\}', raw)
        if json_match:
            try:
                data = json.loads(json_match.group())
                assignments = data.get("section_assignments", {})
                title = data.get("suggested_title", title)
            except json.JSONDecodeError:
                pass

        # Build the report template with finding assignments
        template = ReportTemplate(title=title, metadata={"topic": analysis.topic})
        for section_name in self.STANDARD_SECTIONS:
            assigned_findings = [
                fid for fid, sec in assignments.items() if sec == section_name
            ]
            section = ReportSection(
                heading=section_name,
                citations_used=[
                    sid for f in analysis.findings
                    if f.finding_id in assigned_findings
                    for sid in f.source_ids
                ]
            )
            template.sections.append(section)

        # Ensure all findings are assigned somewhere
        assigned_all = set()
        for fid, sec in assignments.items():
            assigned_all.add(fid)
        unassigned = [f.finding_id for f in analysis.findings if f.finding_id not in assigned_all]
        if unassigned:
            # Put unassigned findings in "Key Findings"
            for section in template.sections:
                if section.heading == "Key Findings":
                    for fid in unassigned:
                        f = next((x for x in analysis.findings if x.finding_id == fid), None)
                        if f:
                            section.citations_used.extend(f.source_ids)

        return template, assignments

# Test the planner
planner = SectionPlanner()
template, assignments = planner.plan(mock_data)

print(f"Report title: {template.title}")
print(f"\nSection plan:")
for section in template.sections:
    assigned = [fid for fid, sec in assignments.items() if sec == section.heading]
    cites = list(set(section.citations_used))
    print(f"  {section.heading:35s} | findings: {assigned} | sources: {cites}")
print(f"\nFinding assignments: {json.dumps(assignments, indent=2)}")

## 4. Section Writer — Writing Individual Sections

The SectionWriter takes a section heading, assigned findings, and source information, then uses the LLM to produce well-structured prose with inline citations.

Each section should have:
- A **topic sentence** that introduces the section's focus
- **Evidence** from the analyzed sources
- **Analysis** connecting findings to the broader narrative
- A **transition** to the next section

In [ ]:
class SectionWriter:
    """Writes individual report sections with citations."""

    def __init__(self, analysis: AnalysisOutput):
        self.analysis = analysis

    def write_section(self, heading: str, finding_ids: List[str],
                      prev_section: str = "", next_heading: str = "") -> str:
        """Write a single section using assigned findings."""
        # Gather the findings for this section
        findings_text = ""
        for fid in finding_ids:
            f = next((x for x in self.analysis.findings if x.finding_id == fid), None)
            if f:
                sources = [self.analysis.get_source(sid) for sid in f.source_ids]
                cite_strs = [s.short_cite() for s in sources if s]
                findings_text += f"- {f.claim} ({', '.join(cite_strs)}). Evidence: {f.evidence}\n"

        # Gather contradictions relevant to this section
        relevant_contradictions = ""
        for c in self.analysis.contradictions:
            if any(fid in c["finding_ids"] for fid in finding_ids):
                relevant_contradictions += f"- {c['description']}\n"

        transition_hint = f"The next section is '{next_heading}'." if next_heading else "This is the final section."

        messages = [
            {"role": "system", "content": "You are an academic writer. Write a report section that:
1. Opens with a clear topic sentence
2. Presents evidence with in-text citations in [Author, Year] format
3. Provides analysis connecting findings
4. Ends with a transition to the next section

Write 2-4 paragraphs. Be precise and professional."},
            {"role": "user", "content": f"Section: {heading}\nTopic: {self.analysis.topic}\n\nFindings for this section:\n{findings_text}\nContradictions to address:\n{relevant_contradictions or 'None'}\n\nTransition: {transition_hint}\n\nWrite the section content (no heading — just the prose):"}
        ]

        content = generate(messages, max_new_tokens=400, temperature=0.7)
        return content.strip()

    def write_abstract(self, all_findings: List[Finding]) -> str:
        """Write an abstract summarizing the entire report."""
        claims = "\n".join(f"- {f.claim}" for f in all_findings)

        messages = [
            {"role": "system", "content": "You are an academic writer. Write a concise abstract (100-150 words) that summarizes the research topic and key findings. Use formal academic tone."},
            {"role": "user", "content": f"Topic: {self.analysis.topic}\n\nKey findings:\n{claims}\n\nWrite the abstract:"}
        ]

        return generate(messages, max_new_tokens=200, temperature=0.5, do_sample=True).strip()

# Test the section writer
writer = SectionWriter(mock_data)
test_section = writer.write_section(
    "Key Findings", ["f1", "f3"],
    next_heading="Analysis and Discussion"
)
print(f"Section: Key Findings\n{'─'*50}")
print(test_section)

## 5. Citation Manager — Tracking and Formatting References

The CitationManager ensures every claim has proper attribution. It tracks which sources are cited, provides in-text citation formatting, and generates the references section.

In [ ]:
class CitationManager:
    """Manages citations throughout the report."""

    def __init__(self, sources: List[Source]):
        self.sources = {s.source_id: s for s in sources}
        self._cited: Dict[str, int] = {}  # source_id → citation count

    def in_text_citation(self, source_id: str) -> str:
        """Get in-text citation for a source: [Author, Year]."""
        source = self.sources.get(source_id)
        if not source:
            return "[Unknown Source]"
        self._cited[source_id] = self._cited.get(source_id, 0) + 1
        return source.short_cite()

    def cite_multiple(self, source_ids: List[str]) -> str:
        """Cite multiple sources: [Author1, Year1; Author2, Year2]."""
        cites = []
        for sid in source_ids:
            source = self.sources.get(sid)
            if source:
                self._cited[sid] = self._cited.get(sid, 0) + 1
                first = source.authors[0].split()[-1] if source.authors else "Unknown"
                et_al = " et al." if len(source.authors) > 2 else ""
                cites.append(f"{first}{et_al}, {source.year}")
        return f"[{'; '.join(cites)}]"

    def generate_references_section(self) -> str:
        """Generate formatted references section, sorted alphabetically."""
        cited_sources = [
            self.sources[sid] for sid in self._cited if sid in self.sources
        ]
        cited_sources.sort(key=lambda s: s.authors[0].split()[-1] if s.authors else "Z")
        lines = []
        for i, source in enumerate(cited_sources, 1):
            lines.append(f"{i}. {source.full_reference()}")
        return "\n".join(lines)

    def get_uncited_sources(self) -> List[Source]:
        """Find sources that exist but haven't been cited."""
        return [
            self.sources[sid] for sid in self.sources if sid not in self._cited
        ]

    def citation_coverage(self) -> float:
        """What fraction of available sources were cited?"""
        if not self.sources:
            return 0.0
        return len(self._cited) / len(self.sources)

    def stats(self) -> Dict[str, Any]:
        """Citation statistics."""
        return {
            "total_sources": len(self.sources),
            "cited_sources": len(self._cited),
            "coverage": f"{self.citation_coverage():.0%}",
            "total_citations": sum(self._cited.values()),
            "most_cited": max(self._cited.items(), key=lambda x: x[1])[0] if self._cited else None,
        }

# Test the citation manager
cm = CitationManager(mock_data.sources)

print("In-text citations:")
print(f"  {cm.in_text_citation('s1')}")
print(f"  {cm.in_text_citation('s2')}")
print(f"  {cm.cite_multiple(['s1', 's3', 's4'])}")
print(f"  {cm.in_text_citation('s1')}")  # second citation of same source

print(f"\nReferences section:")
print(cm.generate_references_section())

print(f"\nUncited: {[s.source_id for s in cm.get_uncited_sources()]}")
print(f"Stats: {cm.stats()}")

## 6. Coherence Checker — Ensuring Quality Across Sections

A report isn't just individual sections pasted together. The CoherenceChecker analyzes the full report for:
- **Redundancy** — are the same points made in multiple sections?
- **Logical flow** — does each section build on the previous?
- **Consistency** — are terms and claims used consistently?

In [ ]:
@dataclass
class CoherenceReport:
    """Results of a coherence check."""
    redundancy_issues: List[str]
    flow_issues: List[str]
    consistency_issues: List[str]
    overall_score: float  # 0-1
    suggestions: List[str]

class CoherenceChecker:
    """Checks report coherence across sections."""

    def check(self, report: ReportTemplate) -> CoherenceReport:
        """Run coherence checks on a report."""
        # Gather all section content
        section_texts = []
        for s in report.sections:
            if s.content:
                section_texts.append(f"[{s.heading}]: {s.content}")

        full_text = "\n\n".join(section_texts)
        if not full_text.strip():
            return CoherenceReport([], [], [], 0.0, ["Report has no content yet."])

        messages = [
            {"role": "system", "content": """You are a report quality checker. Analyze the report for:
1. REDUNDANCY: Are any points repeated across sections?
2. FLOW: Does each section logically follow from the previous?
3. CONSISTENCY: Are terms and claims used consistently?

Respond in JSON:
{"redundancy_issues": ["..."], "flow_issues": ["..."], "consistency_issues": ["..."], "overall_score": 0.85, "suggestions": ["..."]}

Score 0.0-1.0 where 1.0 is perfect coherence. Be constructive."""},
            {"role": "user", "content": f"Report title: {report.title}\n\nSections:\n{full_text}\n\nCheck for coherence issues:"}
        ]

        raw = generate(messages, max_new_tokens=400, temperature=0.3, do_sample=True)

        # Parse response
        json_match = re.search(r'\{[\s\S]*\}', raw)
        if json_match:
            try:
                data = json.loads(json_match.group())
                return CoherenceReport(
                    redundancy_issues=data.get("redundancy_issues", []),
                    flow_issues=data.get("flow_issues", []),
                    consistency_issues=data.get("consistency_issues", []),
                    overall_score=float(data.get("overall_score", 0.7)),
                    suggestions=data.get("suggestions", [])
                )
            except (json.JSONDecodeError, ValueError):
                pass

        return CoherenceReport(
            redundancy_issues=[], flow_issues=[], consistency_issues=[],
            overall_score=0.7,
            suggestions=["Could not parse coherence check — manual review recommended."]
        )

print("✓ CoherenceChecker defined")

## 7. The Full Writing Agent

Now we assemble everything into the `WritingAgent` class. It orchestrates the full pipeline:

```
Analysis Output
      │
      ▼
┌─────────────┐     ┌─────────────┐     ┌─────────────┐
│   Section   │────►│   Section   │────►│  Citation   │
│   Planner   │     │   Writer    │     │  Manager    │
└─────────────┘     └─────────────┘     └─────────────┘
                          │
                          ▼
                    ┌─────────────┐     ┌─────────────┐
                    │  Coherence  │────►│  Iterative  │
                    │  Checker    │     │  Refinement │
                    └─────────────┘     └─────────────┘
                                              │
                                              ▼
                                        Final Report
```

In [ ]:
class WritingAgent:
    """The full Writing Agent — plans, writes, cites, checks, and refines."""

    def __init__(self, max_refinement_rounds: int = 1):
        self.max_refinement_rounds = max_refinement_rounds
        self.planner = SectionPlanner()
        self.checker = CoherenceChecker()
        self.log: List[str] = []

    def _log(self, msg: str):
        self.log.append(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")
        print(f"  ▸ {msg}")

    def generate_report(self, analysis: AnalysisOutput) -> Tuple[ReportTemplate, Dict]:
        """Generate a complete research report from analysis output."""
        metrics = {"stages": {}, "refinement_rounds": 0}
        self.log = []

        # --- Stage 1: Plan ---
        self._log("Stage 1: Planning report structure...")
        start = time.time()
        template, assignments = self.planner.plan(analysis)
        metrics["stages"]["planning"] = round(time.time() - start, 2)
        self._log(f"Title: {template.title}")
        self._log(f"Sections planned: {len(template.sections)}")

        # --- Stage 2: Write abstract ---
        self._log("Stage 2: Writing abstract...")
        start = time.time()
        section_writer = SectionWriter(analysis)
        template.abstract = section_writer.write_abstract(analysis.findings)
        metrics["stages"]["abstract"] = round(time.time() - start, 2)

        # --- Stage 3: Write each section ---
        self._log("Stage 3: Writing sections...")
        start = time.time()
        citation_manager = CitationManager(analysis.sources)

        for i, section in enumerate(template.sections):
            # Find findings assigned to this section
            assigned = [fid for fid, sec in assignments.items() if sec == section.heading]
            if not assigned:
                # For sections without explicit assignments, provide context
                if section.heading == "Introduction":
                    assigned = [analysis.findings[0].finding_id] if analysis.findings else []
                elif section.heading == "Conclusion":
                    assigned = [f.finding_id for f in analysis.findings[-2:]]

            next_heading = template.sections[i+1].heading if i < len(template.sections)-1 else ""
            prev_content = template.sections[i-1].content if i > 0 else ""

            if assigned or section.heading in ["Introduction", "Conclusion"]:
                self._log(f"  Writing: {section.heading} (findings: {assigned})")
                section.content = section_writer.write_section(
                    section.heading, assigned,
                    prev_section=prev_content[:200],
                    next_heading=next_heading
                )
                # Track citations used
                for fid in assigned:
                    f = next((x for x in analysis.findings if x.finding_id == fid), None)
                    if f:
                        for sid in f.source_ids:
                            citation_manager.in_text_citation(sid)
                            if sid not in section.citations_used:
                                section.citations_used.append(sid)

        metrics["stages"]["writing"] = round(time.time() - start, 2)

        # --- Stage 4: Add references ---
        self._log("Stage 4: Generating references...")
        refs_section = ReportSection(
            heading="References",
            content=citation_manager.generate_references_section()
        )
        template.sections.append(refs_section)
        template.citations = list(citation_manager._cited.keys())

        # --- Stage 5: Coherence check + refinement ---
        self._log("Stage 5: Checking coherence...")
        start = time.time()
        coherence = self.checker.check(template)
        metrics["stages"]["coherence"] = round(time.time() - start, 2)
        self._log(f"  Coherence score: {coherence.overall_score:.2f}")

        if coherence.suggestions:
            self._log(f"  Suggestions: {len(coherence.suggestions)}")

        # Iterative refinement if coherence is low
        if coherence.overall_score < 0.75 and self.max_refinement_rounds > 0:
            self._log("Stage 5b: Refining for coherence...")
            start = time.time()
            for round_num in range(self.max_refinement_rounds):
                metrics["refinement_rounds"] += 1
                self._log(f"  Refinement round {round_num + 1}...")
                # Pick the section with most issues and rewrite it
                for section in template.sections:
                    if section.heading == "References":
                        continue
                    if not section.content:
                        continue
                    assigned = [fid for fid, sec in assignments.items() if sec == section.heading]
                    if assigned:
                        section.content = section_writer.write_section(
                            section.heading, assigned
                        )
                # Re-check coherence
                coherence = self.checker.check(template)
                self._log(f"  Updated coherence: {coherence.overall_score:.2f}")
                if coherence.overall_score >= 0.75:
                    break
            metrics["stages"]["refinement"] = round(time.time() - start, 2)

        # --- Compute metrics ---
        metrics["word_count"] = template.word_count()
        metrics["citation_stats"] = citation_manager.stats()
        metrics["coherence_score"] = coherence.overall_score
        metrics["sections_written"] = sum(1 for s in template.sections if s.content)
        metrics["total_time"] = sum(metrics["stages"].values())

        self._log(f"\nReport complete! {metrics['word_count']} words, {metrics['sections_written']} sections")
        return template, metrics

print("✓ WritingAgent defined")

## 8. Test: Generate a Complete Research Report

Let's run the full pipeline. The Writing Agent will:
1. Plan the report structure
2. Write an abstract
3. Write each section with citations
4. Check coherence
5. Refine if needed

In [ ]:
agent = WritingAgent(max_refinement_rounds=1)
report, metrics = agent.generate_report(mock_data)

print("\n" + "="*60)
print("  GENERATION METRICS")
print("="*60)
for stage, elapsed in metrics["stages"].items():
    print(f"  {stage:20s}: {elapsed:5.1f}s")
print(f"  {'total':20s}: {metrics['total_time']:5.1f}s")
print(f"\n  Word count:    {metrics['word_count']}")
print(f"  Sections:      {metrics['sections_written']}")
print(f"  Coherence:     {metrics['coherence_score']:.2f}")
print(f"  Citations:     {metrics['citation_stats']}")

### 8.1 — The Generated Report

Let's view the full rendered report.

In [ ]:
rendered = report.render()
print(rendered)

### 8.2 — Testing with Different Topics

Let's verify the Writing Agent generalizes by generating reports for varied topics.

In [ ]:
def create_mock_for_topic(topic: str, num_findings: int = 4) -> AnalysisOutput:
    """Create a simpler mock dataset for testing different topics."""
    sources = [
        Source(f"src_{i}", f"Study {i} on {topic}",
               [f"Author{i}A", f"Author{i}B"], 2022 + (i % 3),
               source_type="paper")
        for i in range(1, num_findings + 2)
    ]
    findings = [
        Finding(
            f"f{i}",
            claim=f"Finding {i} about {topic}",
            evidence=f"Evidence for finding {i} based on empirical results.",
            source_ids=[f"src_{i}", f"src_{i+1}"],
            confidence=0.7 + (i * 0.05),
            theme=["effectiveness", "scaling", "limitations"][i % 3]
        )
        for i in range(1, num_findings + 1)
    ]
    return AnalysisOutput(topic=topic, findings=findings, sources=sources)

topics = [
    "Retrieval-augmented generation for knowledge-intensive tasks",
    "Multi-agent collaboration in software engineering",
    "Constitutional AI and alignment techniques",
]

for topic in topics:
    print(f"\n{'='*60}")
    print(f"  TOPIC: {topic}")
    print(f"{'='*60}")
    mock = create_mock_for_topic(topic)
    agent2 = WritingAgent(max_refinement_rounds=0)
    report2, metrics2 = agent2.generate_report(mock)
    print(f"\n  Result: {metrics2['word_count']} words, "
          f"{metrics2['sections_written']} sections, "
          f"coherence={metrics2['coherence_score']:.2f}, "
          f"time={metrics2['total_time']:.1f}s")

## 9. Quality Metrics — Measuring Report Quality

We define four quality dimensions and measure them systematically:

| Metric | What It Measures | How We Compute It |
|---|---|---|
| **Completeness** | Are all findings included? | Findings referenced ÷ total findings |
| **Citation coverage** | Are sources properly cited? | Sources cited ÷ total sources |
| **Coherence** | Do sections flow together? | LLM-based coherence check |
| **Readability** | Is the prose clear? | Average words per sentence, vocabulary diversity |

In [ ]:
def compute_quality_metrics(report: ReportTemplate, analysis: AnalysisOutput,
                             metrics: Dict) -> Dict[str, Any]:
    """Compute comprehensive quality metrics for a generated report."""
    quality = {}

    # 1. Completeness — are all findings represented?
    all_content = report.abstract + " " + " ".join(
        s.content for s in report.sections if s.content
    )
    findings_mentioned = 0
    for f in analysis.findings:
        # Check if key words from the claim appear in the report
        key_words = [w.lower() for w in f.claim.split() if len(w) > 4]
        matches = sum(1 for w in key_words if w in all_content.lower())
        if matches >= len(key_words) * 0.3:
            findings_mentioned += 1
    quality["completeness"] = findings_mentioned / len(analysis.findings) if analysis.findings else 0

    # 2. Citation coverage
    quality["citation_coverage"] = metrics.get("citation_stats", {}).get("coverage", "0%")

    # 3. Coherence score (from checker)
    quality["coherence"] = metrics.get("coherence_score", 0)

    # 4. Readability metrics
    sentences = re.split(r'[.!?]+', all_content)
    sentences = [s.strip() for s in sentences if s.strip()]
    words = all_content.split()
    unique_words = set(w.lower() for w in words)

    quality["total_words"] = len(words)
    quality["total_sentences"] = len(sentences)
    quality["avg_words_per_sentence"] = round(len(words) / max(len(sentences), 1), 1)
    quality["vocabulary_diversity"] = round(len(unique_words) / max(len(words), 1), 3)
    quality["sections_with_content"] = sum(1 for s in report.sections if s.content)

    # Overall quality score (weighted average)
    completeness_score = quality["completeness"]
    coverage_val = float(quality["citation_coverage"].strip('%')) / 100 if isinstance(quality["citation_coverage"], str) else quality["citation_coverage"]
    readability_score = min(1.0, 1.0 - abs(quality["avg_words_per_sentence"] - 20) / 30)

    quality["overall_score"] = round(
        0.30 * completeness_score +
        0.25 * coverage_val +
        0.25 * quality["coherence"] +
        0.20 * max(readability_score, 0), 3
    )

    return quality

quality = compute_quality_metrics(report, mock_data, metrics)

print("╔" + "═"*50 + "╗")
print(f"║  QUALITY METRICS" + " "*33 + "║")
print("╠" + "═"*50 + "╣")
for key, value in quality.items():
    if isinstance(value, float):
        line = f"  {key:30s}: {value:.3f}"
    else:
        line = f"  {key:30s}: {value}"
    print(f"║{line:50s}║")
print("╚" + "═"*50 + "╝")

### 9.1 — Quality Comparison: Single-Shot vs. Planned Writing

Let's compare the Writing Agent's structured approach against a single-shot LLM generation to see the benefit of planning and sectioning.

In [ ]:
# Single-shot baseline: ask the LLM to write the full report in one go
finding_text = "\n".join(f"- {f.claim}" for f in mock_data.findings)
source_text = "\n".join(f"- {s.short_cite()} {s.title}" for s in mock_data.sources)

messages = [
    {"role": "system", "content": "You are a research report writer. Write a structured academic report with sections: Introduction, Background, Findings, Analysis, Conclusion, and References."},
    {"role": "user", "content": f"Topic: {mock_data.topic}\n\nKey findings:\n{finding_text}\n\nSources:\n{source_text}\n\nWrite the complete report:"}
]

start = time.time()
single_shot = generate(messages, max_new_tokens=800, temperature=0.7)
single_shot_time = time.time() - start

# Create a simple template for metrics comparison
single_report = ReportTemplate(
    title="Single-shot report",
    abstract="",
    sections=[ReportSection(heading="Full Content", content=single_shot)]
)
single_metrics = {"coherence_score": 0.5, "citation_stats": {"coverage": "0%"}}

sq = compute_quality_metrics(single_report, mock_data, single_metrics)

print("Comparison: Writing Agent vs. Single-Shot")
print("─" * 55)
print(f"{'Metric':30s} | {'Agent':10s} | {'Single-Shot':10s}")
print("─" * 55)
comparisons = [
    ("Total words", quality["total_words"], sq["total_words"]),
    ("Sections with content", quality["sections_with_content"], sq["sections_with_content"]),
    ("Completeness", f"{quality['completeness']:.2f}", f"{sq['completeness']:.2f}"),
    ("Citation coverage", quality["citation_coverage"], sq["citation_coverage"]),
    ("Coherence", f"{quality['coherence']:.2f}", f"{sq['coherence']:.2f}"),
    ("Vocabulary diversity", f"{quality['vocabulary_diversity']:.3f}", f"{sq['vocabulary_diversity']:.3f}"),
    ("Avg words/sentence", quality["avg_words_per_sentence"], sq["avg_words_per_sentence"]),
    ("Overall score", f"{quality['overall_score']:.3f}", f"{sq['overall_score']:.3f}"),
    ("Generation time (s)", f"{metrics['total_time']:.1f}", f"{single_shot_time:.1f}"),
]
for name, agent_val, single_val in comparisons:
    print(f"{name:30s} | {str(agent_val):10s} | {str(single_val):10s}")

print("\n💡 The structured approach trades generation time for better coverage and coherence.")

### 9.2 — Execution Log Review

The Writing Agent logged every step. Let's review the full execution trace.

In [ ]:
print("Writing Agent Execution Log:")
print("═" * 50)
for entry in agent.log:
    print(f"  {entry}")

## 10. Summary and Key Takeaways

### What We Built

| Component | Purpose | Key Methods |
|---|---|---|
| **ReportTemplate** | Data model for structured reports | render(), word_count() |
| **Source / Finding** | Citation and evidence data models | short_cite(), full_reference() |
| **SectionPlanner** | Maps findings to report sections | plan() → outline + assignments |
| **SectionWriter** | Writes individual sections with citations | write_section(), write_abstract() |
| **CitationManager** | Tracks and formats citations | in_text_citation(), generate_references_section() |
| **CoherenceChecker** | Validates cross-section quality | check() → CoherenceReport |
| **WritingAgent** | Orchestrates the full pipeline | generate_report() → report + metrics |

### Design Principles

1. **Structured generation** — don't ask an LLM to write everything at once; plan sections, write them individually, then assemble
2. **Citation-first writing** — every claim traces back to a source
3. **Quality verification** — coherence checking catches redundancy and flow issues
4. **Iterative refinement** — low-coherence reports get rewritten
5. **Measurable quality** — completeness, coverage, coherence, and readability are quantified

### Castalia Scholar Pipeline So Far

| Stage | Notebook | Status |
|---|---|---|
| Research Agent | 34 | ✅ Built |
| **Writing Agent** | **35** | **✅ Built (this notebook)** |
| Orchestrator | 36 | Next |

### What's Next

- **Notebook 36**: Castalia Scholar — The Orchestrator ties Research + Analysis + Writing into a single automated pipeline
- The full system will take a query and produce a complete, cited research report end-to-end

---

*The Writing Agent demonstrates that structured generation — planning before writing, citing as you go, and checking quality after — consistently outperforms single-shot report generation.*